# ✈️ AeroGuard TSLM — Notebook 1: Raw Telemetry Exploration & Pre-Check

This notebook provides an interactive environment to inspect and validate the raw **NASA C-MAPSS FD001** turbofan degradation dataset.

### Objectives:
1. Verify raw data existence and schema integrity (`data/raw/train_FD001.txt`).
2. Analyze engine fleet statistics (100 engines, flight lifetime distributions).
3. Contrast the **14 active aerothermal channels** against the **7 zero-variance invariant channels**.
4. Visualize run-to-failure telemetry trajectories and observe the coupled thermodynamic divergence ($Ps_{30} \downarrow$ and $T_{50} \uparrow$).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✔ Project root: {PROJECT_ROOT}")

✔ Project root: /home/ubuntu/Project/timelapse


## 1. Load Raw C-MAPSS FD001 Telemetry

NASA C-MAPSS raw text files are whitespace-delimited with 26 columns:
- `unit_number`: Engine ID (1 to 100)
- `time_in_cycles`: Current operational cycle (1 to max cycle)
- `op_setting_1`, `op_setting_2`, `op_setting_3`: Operational flight regimes
- `sensor_1` to `sensor_21`: 21 turbofan telemetry channels

In [3]:
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "train_FD001.txt"
assert RAW_PATH.is_file(), f"Raw file not found at {RAW_PATH}. Run 'python -m scripts.download_data' first."

# Standard C-MAPSS column names
COL_NAMES = ["unit_number", "time_in_cycles", "op_setting_1", "op_setting_2", "op_setting_3"] + [
    f"sensor_{i}" for i in range(1, 22)
]

df_raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COL_NAMES)
print(f"✔ Loaded raw telemetry: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()

✔ Loaded raw telemetry: 20,631 rows, 26 columns


,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## 2. Engine Fleet Statistics & Lifetimes

Every turbofan engine runs until catastrophic mechanical threshold ($RUL = 0$). Let's inspect the lifetime distribution across all 100 units.

In [4]:
lifetimes = df_raw.groupby("unit_number")["time_in_cycles"].max().reset_index()
lifetimes.columns = ["unit_number", "max_cycles"]

print(f"• Total Turbofan Engines: {len(lifetimes)}")
print(f"• Shortest Engine Lifetime: {lifetimes['max_cycles'].min()} cycles (Unit #{lifetimes.loc[lifetimes['max_cycles'].idxmin(), 'unit_number']})")
print(f"• Longest Engine Lifetime:  {lifetimes['max_cycles'].max()} cycles (Unit #{lifetimes.loc[lifetimes['max_cycles'].idxmax(), 'unit_number']})")
print(f"• Fleet Average Lifetime:   {lifetimes['max_cycles'].mean():.2f} cycles")

fig = px.histogram(
    lifetimes,
    x="max_cycles",
    nbins=20,
    title="Fleet Engine Lifetime Distribution (FD001)",
    labels={"max_cycles": "Max Operational Cycles (Lifetime)"},
    color_discrete_sequence=["#3A7BD5"]
)
fig.update_layout(template="plotly_dark", height=380)
fig.show()

• Total Turbofan Engines: 100
• Shortest Engine Lifetime: 128 cycles (Unit #39)
• Longest Engine Lifetime:  362 cycles (Unit #69)
• Fleet Average Lifetime:   206.31 cycles


## 3. Active vs. Invariant Channel Analysis

In the FD001 dataset, NASA simulated sea-level steady-state cruise.
- **7 channels** have standard deviation $\sigma = 0.0$ (ambient invariant) and are dropped to prevent singular covariance matrices.
- **14 channels** are active aerothermal degradation channels across 7 physical engine stations.

In [5]:
ACTIVE_CHANNELS = [
    "sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_8",
    "sensor_9", "sensor_11", "sensor_12", "sensor_13", "sensor_14",
    "sensor_15", "sensor_17", "sensor_20", "sensor_21"
]

all_sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
sensor_stds = df_raw[all_sensor_cols].std()

channel_summary = pd.DataFrame({
    "Channel": all_sensor_cols,
    "Standard_Deviation": sensor_stds.values,
    "Status": ["Active" if c in ACTIVE_CHANNELS else "Dropped (Zero-Variance)" for c in all_sensor_cols]
})

fig = px.bar(
    channel_summary,
    x="Channel",
    y="Standard_Deviation",
    color="Status",
    title="Sensor Channel Variances: 14 Active vs. 7 Dropped Invariant Channels",
    color_discrete_map={"Active": "#00D26A", "Dropped (Zero-Variance)": "#FF4B4B"}
)
fig.update_layout(template="plotly_dark", height=400)
fig.show()

## 4. Run-to-Failure Telemetry Visualization

Let's inspect the physical degradation signature for held-out **Engine Unit #84** (which fails at Cycle 267):
- Notice **Exhaust Gas Temperature ($T_{50}$ / sensor_4)** steadily surging upward.
- Concurrently, **High-Pressure Compressor Static Pressure ($P_{s30}$ / sensor_11)** drops as blade clearances erode.

In [6]:
unit_id = 84
engine_data = df_raw[df_raw["unit_number"] == unit_id].sort_values("time_in_cycles")

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        f"Engine Unit #{unit_id}: Exhaust Gas Temp T50 (Sensor 4 - °R) -> Surges Upward",
        f"Engine Unit #{unit_id}: HPC Static Pressure Ps30 (Sensor 11 - psia) -> Drops Downward"
    ]
)

fig.add_trace(
    go.Scatter(x=engine_data["time_in_cycles"], y=engine_data["sensor_4"], name="T50 (EGT)", line=dict(color="#FF4B4B", width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=engine_data["time_in_cycles"], y=engine_data["sensor_11"], name="Ps30 (HPC Press)", line=dict(color="#00D26A", width=2)),
    row=2, col=1
)

fig.update_layout(template="plotly_dark", height=500, title_text=f"Run-to-Failure Aerothermal Divergence (Engine #{unit_id})")
fig.show()